In [1]:
import os
import sys

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = (
    SparkSession.builder
    .appName("Spark")
    .master("local[*]")
    .getOrCreate()
)

In [4]:
spark

In [5]:
emp_data = [
    ("E101", "D01", "Amit Sharma", "Male", "75000", "2021-03-15"),
    ("E102", "D02", "Priya Mehta", "Female", "82000", "2020-07-21"),
    ("E103", "D01", "Rahul Verma", "Male", "68000", "2022-01-10"),
    ("E104", "D03", "Neha Kapoor", "Female", "91000", "2019-11-05"),
    ("E105", "D02", "Karan Singh", "Male", "72000", "2023-02-14"),
    ("E106", "D04", "Ananya Gupta", "Female", "88000", "2021-08-30"),
    ("E107", "D03", "Vikram Patel", "Male", "95000", "2018-12-12"),
    ("E108", "D01", "Sneha Reddy", "Female", "77000", "2022-06-19"),
    ("E109", "D04", "Arjun Nair", "Male", "83000", "2020-04-27"),
    ("E110", "D02", "Meera Iyer", "Female", "79000", "2023-09-01"),
]

emp_schema = "employee_id string, department_id string, name string, gender string, salary string, hire_date string"

In [6]:
emp =  spark.createDataFrame(data=emp_data, schema=emp_schema)

In [7]:
emp.rdd.getNumPartitions()

8

In [8]:
emp.show()

+-----------+-------------+------------+------+------+----------+
|employee_id|department_id|        name|gender|salary| hire_date|
+-----------+-------------+------------+------+------+----------+
|       E101|          D01| Amit Sharma|  Male| 75000|2021-03-15|
|       E102|          D02| Priya Mehta|Female| 82000|2020-07-21|
|       E103|          D01| Rahul Verma|  Male| 68000|2022-01-10|
|       E104|          D03| Neha Kapoor|Female| 91000|2019-11-05|
|       E105|          D02| Karan Singh|  Male| 72000|2023-02-14|
|       E106|          D04|Ananya Gupta|Female| 88000|2021-08-30|
|       E107|          D03|Vikram Patel|  Male| 95000|2018-12-12|
|       E108|          D01| Sneha Reddy|Female| 77000|2022-06-19|
|       E109|          D04|  Arjun Nair|  Male| 83000|2020-04-27|
|       E110|          D02|  Meera Iyer|Female| 79000|2023-09-01|
+-----------+-------------+------------+------+------+----------+



In [9]:
emp_final = emp.where("salary > 50000")

In [10]:
emp_final.rdd.getNumPartitions()

8

In [11]:
emp_final.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("data/emp_output")

In [12]:
emp.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [13]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [14]:
schema_string = "name string, age int"
schema_spark = StructType([
    StructField("name",StringType(), True),
    StructField("age",IntegerType(), True)
]) 

In [15]:
from pyspark.sql.functions import col, expr

emp['salary']

Column<'salary'>

In [16]:
emp_filter = emp.select(col('employee_id'), expr("name"), emp.salary)

In [17]:
emp_filter.show()

+-----------+------------+------+
|employee_id|        name|salary|
+-----------+------------+------+
|       E101| Amit Sharma| 75000|
|       E102| Priya Mehta| 82000|
|       E103| Rahul Verma| 68000|
|       E104| Neha Kapoor| 91000|
|       E105| Karan Singh| 72000|
|       E106|Ananya Gupta| 88000|
|       E107|Vikram Patel| 95000|
|       E108| Sneha Reddy| 77000|
|       E109|  Arjun Nair| 83000|
|       E110|  Meera Iyer| 79000|
+-----------+------------+------+



In [18]:
emp_filter_rename = emp.select(col('employee_id').alias("emp_id"),emp.name, expr("cast(salary as int)as salary"))

In [19]:
emp_filter_rename.explain(True)

== Parsed Logical Plan ==
'Project ['employee_id AS emp_id#59, name#2, cast('salary as int) AS salary#60]
+- LogicalRDD [employee_id#0, department_id#1, name#2, gender#3, salary#4, hire_date#5], false

== Analyzed Logical Plan ==
emp_id: string, name: string, salary: int
Project [employee_id#0 AS emp_id#59, name#2, cast(salary#4 as int) AS salary#60]
+- LogicalRDD [employee_id#0, department_id#1, name#2, gender#3, salary#4, hire_date#5], false

== Optimized Logical Plan ==
Project [employee_id#0 AS emp_id#59, name#2, cast(salary#4 as int) AS salary#60]
+- LogicalRDD [employee_id#0, department_id#1, name#2, gender#3, salary#4, hire_date#5], false

== Physical Plan ==
*(1) Project [employee_id#0 AS emp_id#59, name#2, cast(salary#4 as int) AS salary#60]
+- *(1) Scan ExistingRDD[employee_id#0,department_id#1,name#2,gender#3,salary#4,hire_date#5]



In [20]:
emp_filter_rename.show()

+------+------------+------+
|emp_id|        name|salary|
+------+------------+------+
|  E101| Amit Sharma| 75000|
|  E102| Priya Mehta| 82000|
|  E103| Rahul Verma| 68000|
|  E104| Neha Kapoor| 91000|
|  E105| Karan Singh| 72000|
|  E106|Ananya Gupta| 88000|
|  E107|Vikram Patel| 95000|
|  E108| Sneha Reddy| 77000|
|  E109|  Arjun Nair| 83000|
|  E110|  Meera Iyer| 79000|
+------+------------+------+



In [21]:
emp_filter_rename.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)



In [22]:
emp_filter_1 = emp_filter.selectExpr("employee_id as emp_id"," name", "cast(salary as int) as salary")
emp_filter_1.show()

+------+------------+------+
|emp_id|        name|salary|
+------+------------+------+
|  E101| Amit Sharma| 75000|
|  E102| Priya Mehta| 82000|
|  E103| Rahul Verma| 68000|
|  E104| Neha Kapoor| 91000|
|  E105| Karan Singh| 72000|
|  E106|Ananya Gupta| 88000|
|  E107|Vikram Patel| 95000|
|  E108| Sneha Reddy| 77000|
|  E109|  Arjun Nair| 83000|
|  E110|  Meera Iyer| 79000|
+------+------------+------+



In [23]:
emp_filter_1.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)



In [24]:
emp_salary_filter = emp.selectExpr("*").where("salary>75000")

In [25]:
emp_salary_filter.show()

+-----------+-------------+------------+------+------+----------+
|employee_id|department_id|        name|gender|salary| hire_date|
+-----------+-------------+------------+------+------+----------+
|       E102|          D02| Priya Mehta|Female| 82000|2020-07-21|
|       E104|          D03| Neha Kapoor|Female| 91000|2019-11-05|
|       E106|          D04|Ananya Gupta|Female| 88000|2021-08-30|
|       E107|          D03|Vikram Patel|  Male| 95000|2018-12-12|
|       E108|          D01| Sneha Reddy|Female| 77000|2022-06-19|
|       E109|          D04|  Arjun Nair|  Male| 83000|2020-04-27|
|       E110|          D02|  Meera Iyer|Female| 79000|2023-09-01|
+-----------+-------------+------------+------+------+----------+



In [26]:
emp_salary_filter.write.format("csv").mode("overwrite").save("data/output/emp_salary_filter.csv")

In [27]:
emp.write.format("csv").mode("overwrite").save("data/output/emp.csv")

### Cast Column | Add Column | Static Column Value |Rename

In [28]:
emp.show()

+-----------+-------------+------------+------+------+----------+
|employee_id|department_id|        name|gender|salary| hire_date|
+-----------+-------------+------------+------+------+----------+
|       E101|          D01| Amit Sharma|  Male| 75000|2021-03-15|
|       E102|          D02| Priya Mehta|Female| 82000|2020-07-21|
|       E103|          D01| Rahul Verma|  Male| 68000|2022-01-10|
|       E104|          D03| Neha Kapoor|Female| 91000|2019-11-05|
|       E105|          D02| Karan Singh|  Male| 72000|2023-02-14|
|       E106|          D04|Ananya Gupta|Female| 88000|2021-08-30|
|       E107|          D03|Vikram Patel|  Male| 95000|2018-12-12|
|       E108|          D01| Sneha Reddy|Female| 77000|2022-06-19|
|       E109|          D04|  Arjun Nair|  Male| 83000|2020-04-27|
|       E110|          D02|  Meera Iyer|Female| 79000|2023-09-01|
+-----------+-------------+------------+------+------+----------+



In [29]:
emp.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [30]:
from pyspark.sql.functions import col, cast

In [31]:
emp_casted = emp.select("employee_id", "name", col("salary").cast("double"))
emp_casted.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: double (nullable = true)



In [32]:
emp_taxed = emp_casted.withColumn("tax",col("salary")*0.2)

In [33]:
emp_taxed.show()

+-----------+------------+-------+-------+
|employee_id|        name| salary|    tax|
+-----------+------------+-------+-------+
|       E101| Amit Sharma|75000.0|15000.0|
|       E102| Priya Mehta|82000.0|16400.0|
|       E103| Rahul Verma|68000.0|13600.0|
|       E104| Neha Kapoor|91000.0|18200.0|
|       E105| Karan Singh|72000.0|14400.0|
|       E106|Ananya Gupta|88000.0|17600.0|
|       E107|Vikram Patel|95000.0|19000.0|
|       E108| Sneha Reddy|77000.0|15400.0|
|       E109|  Arjun Nair|83000.0|16600.0|
|       E110|  Meera Iyer|79000.0|15800.0|
+-----------+------------+-------+-------+



### Add  static columns

In [34]:
from pyspark.sql.functions import lit

In [35]:
emp_new_cols = emp_taxed.withColumn("COL_ONE",lit(1)).withColumn("COL_TWO",lit(2))

In [36]:
emp_new_cols.show()

+-----------+------------+-------+-------+-------+-------+
|employee_id|        name| salary|    tax|COL_ONE|COL_TWO|
+-----------+------------+-------+-------+-------+-------+
|       E101| Amit Sharma|75000.0|15000.0|      1|      2|
|       E102| Priya Mehta|82000.0|16400.0|      1|      2|
|       E103| Rahul Verma|68000.0|13600.0|      1|      2|
|       E104| Neha Kapoor|91000.0|18200.0|      1|      2|
|       E105| Karan Singh|72000.0|14400.0|      1|      2|
|       E106|Ananya Gupta|88000.0|17600.0|      1|      2|
|       E107|Vikram Patel|95000.0|19000.0|      1|      2|
|       E108| Sneha Reddy|77000.0|15400.0|      1|      2|
|       E109|  Arjun Nair|83000.0|16600.0|      1|      2|
|       E110|  Meera Iyer|79000.0|15800.0|      1|      2|
+-----------+------------+-------+-------+-------+-------+



In [37]:
emp_new_cols.explain(True)

== Parsed Logical Plan ==
Project [employee_id#0, name#2, salary#163, tax#167, COL_ONE#189, 2 AS COL_TWO#195]
+- Project [employee_id#0, name#2, salary#163, tax#167, 1 AS COL_ONE#189]
   +- Project [employee_id#0, name#2, salary#163, (salary#163 * 0.2) AS tax#167]
      +- Project [employee_id#0, name#2, cast(salary#4 as double) AS salary#163]
         +- LogicalRDD [employee_id#0, department_id#1, name#2, gender#3, salary#4, hire_date#5], false

== Analyzed Logical Plan ==
employee_id: string, name: string, salary: double, tax: double, COL_ONE: int, COL_TWO: int
Project [employee_id#0, name#2, salary#163, tax#167, COL_ONE#189, 2 AS COL_TWO#195]
+- Project [employee_id#0, name#2, salary#163, tax#167, 1 AS COL_ONE#189]
   +- Project [employee_id#0, name#2, salary#163, (salary#163 * 0.2) AS tax#167]
      +- Project [employee_id#0, name#2, cast(salary#4 as double) AS salary#163]
         +- LogicalRDD [employee_id#0, department_id#1, name#2, gender#3, salary#4, hire_date#5], false

== Op

### Rename columns

In [38]:
emp_rename_cols = emp_new_cols.withColumnRenamed("COL_ONE","Col_One")
emp_rename_cols.show()

+-----------+------------+-------+-------+-------+-------+
|employee_id|        name| salary|    tax|Col_One|COL_TWO|
+-----------+------------+-------+-------+-------+-------+
|       E101| Amit Sharma|75000.0|15000.0|      1|      2|
|       E102| Priya Mehta|82000.0|16400.0|      1|      2|
|       E103| Rahul Verma|68000.0|13600.0|      1|      2|
|       E104| Neha Kapoor|91000.0|18200.0|      1|      2|
|       E105| Karan Singh|72000.0|14400.0|      1|      2|
|       E106|Ananya Gupta|88000.0|17600.0|      1|      2|
|       E107|Vikram Patel|95000.0|19000.0|      1|      2|
|       E108| Sneha Reddy|77000.0|15400.0|      1|      2|
|       E109|  Arjun Nair|83000.0|16600.0|      1|      2|
|       E110|  Meera Iyer|79000.0|15800.0|      1|      2|
+-----------+------------+-------+-------+-------+-------+



## cols renamed with spaces


In [39]:
emp_rename_cols_1 = emp_new_cols.withColumnRenamed("COL_TWO","Col Two")
emp_rename_cols_1.show()

+-----------+------------+-------+-------+-------+-------+
|employee_id|        name| salary|    tax|COL_ONE|Col Two|
+-----------+------------+-------+-------+-------+-------+
|       E101| Amit Sharma|75000.0|15000.0|      1|      2|
|       E102| Priya Mehta|82000.0|16400.0|      1|      2|
|       E103| Rahul Verma|68000.0|13600.0|      1|      2|
|       E104| Neha Kapoor|91000.0|18200.0|      1|      2|
|       E105| Karan Singh|72000.0|14400.0|      1|      2|
|       E106|Ananya Gupta|88000.0|17600.0|      1|      2|
|       E107|Vikram Patel|95000.0|19000.0|      1|      2|
|       E108| Sneha Reddy|77000.0|15400.0|      1|      2|
|       E109|  Arjun Nair|83000.0|16600.0|      1|      2|
|       E110|  Meera Iyer|79000.0|15800.0|      1|      2|
+-----------+------------+-------+-------+-------+-------+



### drop cols


In [40]:
emp_dropped = emp_new_cols.drop("COL_TWO")
emp_dropped.show()

+-----------+------------+-------+-------+-------+
|employee_id|        name| salary|    tax|COL_ONE|
+-----------+------------+-------+-------+-------+
|       E101| Amit Sharma|75000.0|15000.0|      1|
|       E102| Priya Mehta|82000.0|16400.0|      1|
|       E103| Rahul Verma|68000.0|13600.0|      1|
|       E104| Neha Kapoor|91000.0|18200.0|      1|
|       E105| Karan Singh|72000.0|14400.0|      1|
|       E106|Ananya Gupta|88000.0|17600.0|      1|
|       E107|Vikram Patel|95000.0|19000.0|      1|
|       E108| Sneha Reddy|77000.0|15400.0|      1|
|       E109|  Arjun Nair|83000.0|16600.0|      1|
|       E110|  Meera Iyer|79000.0|15800.0|      1|
+-----------+------------+-------+-------+-------+



### limit

In [42]:
emp_limit = emp_new_cols.limit(5)
emp_limit.show()

+-----------+-----------+-------+-------+-------+-------+
|employee_id|       name| salary|    tax|COL_ONE|COL_TWO|
+-----------+-----------+-------+-------+-------+-------+
|       E101|Amit Sharma|75000.0|15000.0|      1|      2|
|       E102|Priya Mehta|82000.0|16400.0|      1|      2|
|       E103|Rahul Verma|68000.0|13600.0|      1|      2|
|       E104|Neha Kapoor|91000.0|18200.0|      1|      2|
|       E105|Karan Singh|72000.0|14400.0|      1|      2|
+-----------+-----------+-------+-------+-------+-------+



### adding multiple columns 

In [45]:
emp.show()

+-----------+-------------+------------+------+------+----------+
|employee_id|department_id|        name|gender|salary| hire_date|
+-----------+-------------+------------+------+------+----------+
|       E101|          D01| Amit Sharma|  Male| 75000|2021-03-15|
|       E102|          D02| Priya Mehta|Female| 82000|2020-07-21|
|       E103|          D01| Rahul Verma|  Male| 68000|2022-01-10|
|       E104|          D03| Neha Kapoor|Female| 91000|2019-11-05|
|       E105|          D02| Karan Singh|  Male| 72000|2023-02-14|
|       E106|          D04|Ananya Gupta|Female| 88000|2021-08-30|
|       E107|          D03|Vikram Patel|  Male| 95000|2018-12-12|
|       E108|          D01| Sneha Reddy|Female| 77000|2022-06-19|
|       E109|          D04|  Arjun Nair|  Male| 83000|2020-04-27|
|       E110|          D02|  Meera Iyer|Female| 79000|2023-09-01|
+-----------+-------------+------------+------+------+----------+



In [46]:
cols = {
    'tax' : col('salary') * 0.2,
    "num_one": lit(1),
    "num_two": lit("two")
}

emp_mul_cols = emp.withColumns(cols)
emp_mul_cols.show()

+-----------+-------------+------------+------+------+----------+-------+-------+-------+
|employee_id|department_id|        name|gender|salary| hire_date|    tax|num_one|num_two|
+-----------+-------------+------------+------+------+----------+-------+-------+-------+
|       E101|          D01| Amit Sharma|  Male| 75000|2021-03-15|15000.0|      1|    two|
|       E102|          D02| Priya Mehta|Female| 82000|2020-07-21|16400.0|      1|    two|
|       E103|          D01| Rahul Verma|  Male| 68000|2022-01-10|13600.0|      1|    two|
|       E104|          D03| Neha Kapoor|Female| 91000|2019-11-05|18200.0|      1|    two|
|       E105|          D02| Karan Singh|  Male| 72000|2023-02-14|14400.0|      1|    two|
|       E106|          D04|Ananya Gupta|Female| 88000|2021-08-30|17600.0|      1|    two|
|       E107|          D03|Vikram Patel|  Male| 95000|2018-12-12|19000.0|      1|    two|
|       E108|          D01| Sneha Reddy|Female| 77000|2022-06-19|15400.0|      1|    two|
|       E1

### case when

In [47]:
from pyspark.sql.functions import col, when

In [48]:
emp_gender_fixed = emp.withColumn('new_gender',when(col('gender') == "Male",'M')
                                .when(col('gender')=="Female",'F')
                                .otherwise(None)
                                )
    
emp_gender_fixed.show()

+-----------+-------------+------------+------+------+----------+----------+
|employee_id|department_id|        name|gender|salary| hire_date|new_gender|
+-----------+-------------+------------+------+------+----------+----------+
|       E101|          D01| Amit Sharma|  Male| 75000|2021-03-15|         M|
|       E102|          D02| Priya Mehta|Female| 82000|2020-07-21|         F|
|       E103|          D01| Rahul Verma|  Male| 68000|2022-01-10|         M|
|       E104|          D03| Neha Kapoor|Female| 91000|2019-11-05|         F|
|       E105|          D02| Karan Singh|  Male| 72000|2023-02-14|         M|
|       E106|          D04|Ananya Gupta|Female| 88000|2021-08-30|         F|
|       E107|          D03|Vikram Patel|  Male| 95000|2018-12-12|         M|
|       E108|          D01| Sneha Reddy|Female| 77000|2022-06-19|         F|
|       E109|          D04|  Arjun Nair|  Male| 83000|2020-04-27|         M|
|       E110|          D02|  Meera Iyer|Female| 79000|2023-09-01|         F|

In [52]:
from pyspark.sql.functions import col, when

emp_gender_fixed_1 = emp.withColumn(
    "gender",
    when(col("gender") == "Male", "M")
    .when(col("gender") == "Female", "F")
    .otherwise(None)
)

emp_gender_fixed_1.show()

+-----------+-------------+------------+------+------+----------+
|employee_id|department_id|        name|gender|salary| hire_date|
+-----------+-------------+------------+------+------+----------+
|       E101|          D01| Amit Sharma|     M| 75000|2021-03-15|
|       E102|          D02| Priya Mehta|     F| 82000|2020-07-21|
|       E103|          D01| Rahul Verma|     M| 68000|2022-01-10|
|       E104|          D03| Neha Kapoor|     F| 91000|2019-11-05|
|       E105|          D02| Karan Singh|     M| 72000|2023-02-14|
|       E106|          D04|Ananya Gupta|     F| 88000|2021-08-30|
|       E107|          D03|Vikram Patel|     M| 95000|2018-12-12|
|       E108|          D01| Sneha Reddy|     F| 77000|2022-06-19|
|       E109|          D04|  Arjun Nair|     M| 83000|2020-04-27|
|       E110|          D02|  Meera Iyer|     F| 79000|2023-09-01|
+-----------+-------------+------------+------+------+----------+

